In [ ]:
#? Normal class in py 

class Pn:
    def __init__(self, name , age):
        self.name=name
        self.age=age

person=Pn(name="John" , age=21)

print(person.name)
print(person.age)

John
21


## SQLALCHEMY DATA CLASS

In [2]:
from sqlalchemy.orm import DeclarativeBase ,Mapped ,mapped_column 
from sqlalchemy import create_engine

class Base(DeclarativeBase):
    pass

class Person(Base):
    __tablename__="people"

    id:Mapped[int] = mapped_column(primary_key=True)

    name:Mapped[str] = mapped_column(nullable=False)

    age:Mapped[int]=mapped_column(nullable=False)

engine=create_engine('sqlite:///orm_eg.db' , echo=True)
Base.metadata.create_all(engine)

2026-09-12 07:25:58,030 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-12 07:25:58,031 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("people")
2026-09-12 07:25:58,031 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-12 07:25:58,032 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("people")
2026-09-12 07:25:58,032 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-12 07:25:58,034 INFO sqlalchemy.engine.Engine 
CREATE TABLE people (
	id INTEGER NOT NULL, 
	name VARCHAR NOT NULL, 
	age INTEGER NOT NULL, 
	PRIMARY KEY (id)
)


2026-09-12 07:25:58,035 INFO sqlalchemy.engine.Engine [no key 0.00085s] ()
2026-09-12 07:25:58,044 INFO sqlalchemy.engine.Engine COMMIT


## SESSIONS

In [8]:
from sqlalchemy.orm import Session

with Session(engine) as session:
    # 1. Create a list of individual Person objects
    people = [
        Person(name="Jason", age=22),
        Person(name="Kaleb", age=29),
        Person(name="Rick", age=23)
    ]
    
    # 2. Use add_all to stage all of them
    session.add_all(people)
    session.commit()


2026-09-12 07:38:11,401 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-12 07:38:11,402 INFO sqlalchemy.engine.Engine INSERT INTO people (name, age) VALUES (?, ?) RETURNING id
2026-09-12 07:38:11,403 INFO sqlalchemy.engine.Engine [generated in 0.00008s (insertmanyvalues) 1/3 (ordered; batch not supported)] ('Jason', 22)
2026-09-12 07:38:11,404 INFO sqlalchemy.engine.Engine INSERT INTO people (name, age) VALUES (?, ?) RETURNING id
2026-09-12 07:38:11,404 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/3 (ordered; batch not supported)] ('Kaleb', 29)
2026-09-12 07:38:11,405 INFO sqlalchemy.engine.Engine INSERT INTO people (name, age) VALUES (?, ?) RETURNING id
2026-09-12 07:38:11,405 INFO sqlalchemy.engine.Engine [insertmanyvalues 3/3 (ordered; batch not supported)] ('Rick', 23)
2026-09-12 07:38:11,406 INFO sqlalchemy.engine.Engine COMMIT


In [9]:
#! Update in ORM

with Session(engine) as session:
    person = session.get(Person , 3)
    person.age=31
    session.commit()

2026-09-12 07:38:42,861 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-12 07:38:42,862 INFO sqlalchemy.engine.Engine SELECT people.id AS people_id, people.name AS people_name, people.age AS people_age 
FROM people 
WHERE people.id = ?
2026-09-12 07:38:42,862 INFO sqlalchemy.engine.Engine [cached since 370.1s ago] (3,)
2026-09-12 07:38:42,863 INFO sqlalchemy.engine.Engine UPDATE people SET age=? WHERE people.id = ?
2026-09-12 07:38:42,864 INFO sqlalchemy.engine.Engine [cached since 370.1s ago] (31, 3)
2026-09-12 07:38:42,864 INFO sqlalchemy.engine.Engine COMMIT


In [18]:
from sqlalchemy import create_engine, inspect
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy import String


class Base(DeclarativeBase):
    pass


class Book(Base):
    __tablename__ = "books"

    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(100))


engine = create_engine(
    "sqlite:///states.db",
    echo=True
)

Base.metadata.create_all(engine)

with Session(engine) as session:

    book = Book(title="Fight Club")

    print("1:", inspect(book).transient)

    session.add(book)

    print("2:", inspect(book).pending)

    session.commit()

    print("3:", inspect(book).persistent)

print("4:", inspect(book).detached)

2026-09-12 11:33:33,570 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-12 11:33:33,574 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("books")
2026-09-12 11:33:33,575 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-12 11:33:33,577 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("books")
2026-09-12 11:33:33,579 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-12 11:33:33,583 INFO sqlalchemy.engine.Engine 
CREATE TABLE books (
	id INTEGER NOT NULL, 
	title VARCHAR(100) NOT NULL, 
	PRIMARY KEY (id)
)


2026-09-12 11:33:33,586 INFO sqlalchemy.engine.Engine [no key 0.00210s] ()
2026-09-12 11:33:33,599 INFO sqlalchemy.engine.Engine COMMIT
1: True
2: True
2026-09-12 11:33:33,603 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-12 11:33:33,606 INFO sqlalchemy.engine.Engine INSERT INTO books (title) VALUES (?)
2026-09-12 11:33:33,608 INFO sqlalchemy.engine.Engine [generated in 0.00156s] ('Fight Club',)
2026-09-12 11:33:33,612 INFO sqlalchemy.engine.Engine COMMIT
3:

## SESSION MAKER

In [6]:
 #! THINK SESSION MAKER AS A SESSION FACTORY

from sqlalchemy import create_engine , String

from sqlalchemy.orm import DeclarativeBase , Mapped , mapped_column , sessionmaker

class Base(DeclarativeBase):
    pass

class Book(Base):
    __tablename__="Book1"

    id:Mapped[int]=mapped_column(primary_key=True)

    title:Mapped[str]=mapped_column(nullable=False)

    author:Mapped[str]=mapped_column(nullable=False)

engine=create_engine("sqlite:///session.db", echo=True)

Base.metadata.create_all(engine)

SessionLocal=sessionmaker(bind=engine) #? WHAT THE PARAMS DO HERE "bind: Sessions created by this factory should use this Engine. "

with SessionLocal() as session:

    book1 = Book(
        title="Book 1",
        author="Author 1"
    )

    book2 = Book(
        title="Book 2",
        author="Author 2"
    )

    session.add(book1)
    session.add(book2)

    try:
        session.commit()

    except Exception:
        session.rollback()

session.close()

2026-09-12 21:22:27,452 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-12 21:22:27,453 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("Book1")
2026-09-12 21:22:27,454 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-12 21:22:27,455 INFO sqlalchemy.engine.Engine COMMIT
2026-09-12 21:22:27,456 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-12 21:22:27,457 INFO sqlalchemy.engine.Engine INSERT INTO "Book1" (title, author) VALUES (?, ?) RETURNING id
2026-09-12 21:22:27,457 INFO sqlalchemy.engine.Engine [generated in 0.00008s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('Book 1', 'Author 1')
2026-09-12 21:22:27,458 INFO sqlalchemy.engine.Engine INSERT INTO "Book1" (title, author) VALUES (?, ?) RETURNING id
2026-09-12 21:22:27,458 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2 (ordered; batch not supported)] ('Book 2', 'Author 2')
2026-09-12 21:22:27,460 INFO sqlalchemy.engine.Engine COMMIT
